### Inference Preprocess Pipeline


### 1. Set Kernel Env


In [ ]:
# %%bash

# PKGs=$(poetry env info --path)/lib/python3.9/site-packages
# echo $PKGs
# echo $PYTHONPATH
# export PYTHONPATH=$PYTHONPATH:$PKGs


In [ ]:
# %pip install -e /home/gorelova_i_v/projects/cvm_churn-from-dac_binary-class_churn-dac


In [ ]:
%load_ext autoreload
%load_ext dotenv
%dotenv
%autoreload 2


### 2. State and Connections


In [ ]:
try:
    spark.stop()
except NameError:
    pass

import logging
from datetime import datetime
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from cvm_model.io import State
import cvm_model.sql as sql
import cvm_model.utils as utils
from cvm_model.inference.preprocess import _prepare_dataset, _validate_dataset
from cvm_model.parameters import (
    aud_table,
    fav_omni_features_table,
    features,
    features_for_outliers,
    inference_data_stat_suffix,
    input_suffix,
    model_predictions_suffix,
    model_type,
    preperiod_months,
    project_name,
    score,
    target,
    template,
)

logging.basicConfig(level=logging.INFO)
pd.set_option('display.max_columns', None)

state = State.from_env()
engine = state.credentials.loyalty_gp.sa_engine
s3 = state.credentials.cvm_s3
session = state.spark.session

import os
os.environ.update(state.credentials.mlflow.environment)
if state.credentials.mlflow.tracking_uri:
    mlflow.set_tracking_uri(state.credentials.mlflow.tracking_uri)

print('MLflow URI:', mlflow.get_tracking_uri())


### 3. Set Event Timestamp


In [ ]:
event_timestamp = datetime(2026, 6, 1)

run_month = pd.Timestamp(event_timestamp.date().replace(day=1).isoformat())
base_month = (run_month - pd.DateOffset(months=1)).date().isoformat()
feature_date = run_month.date().isoformat()
base_month_suffix = base_month.replace('-', '_')

aud_table_month = f'{aud_table}_{base_month_suffix}_inference'
fav_omni_features_table_month = f'{fav_omni_features_table}_{base_month_suffix}_inference'

print('Base DAC month:', base_month)
print('Feature date:', feature_date)
print('Audience table:', aud_table_month)
print('Favorite OMNI table:', fav_omni_features_table_month)


### 4. Resolve S3 Input Path


In [ ]:
input_prefix = state.settings.preprocess_prefix(event_timestamp) / input_suffix / 'inference'
input_bucket = input_prefix.split('//')[1].split('/')[0]
input_prefix = '/'.join(input_prefix.split('//')[1].split('/')[1:])
input_path = template.format(bucket=input_bucket, prefix=input_prefix)

print('Input bucket:', input_bucket)
print('Input prefix:', input_prefix)
print('Input path:', input_path)


### 5. Clean Inference Input Prefix


In [ ]:
utils.remove_s3_prefix(s3, input_bucket, input_prefix)
assert not utils.list_s3_objects(s3, input_bucket, input_prefix), 'S3 inference prefix was not cleaned'
print('Cleaned:', f's3://{input_bucket}/{input_prefix}')


### 6. Audience Table


In [ ]:
full_aud_query = sql.aud_for_scoring_query.format(base_month=base_month)
df = utils.get_df(engine, full_aud_query).fillna(0)
df = df.astype({col: np.int64 for col in {'contact_id'} & set(df.columns)})

assert len(df) > 0, 'No DAC audience contacts were loaded'
assert df['contact_id'].is_unique, 'DAC audience contains duplicate contact_id values'

print('Inference audience shape:', df.shape)
display(df.head())


### 7. Upload Audience to GP


In [ ]:
utils.upload_df(engine, pd.DataFrame(df['contact_id']).astype(np.int64), aud_table_month)
aud_query = f'select distinct contact_id ::bigint as contact_id from {aud_table_month}'

check_df = utils.get_df(engine, sql.check_unique_query.format(aud=aud_query))
display(check_df)


### 8. Load Features


In [ ]:
df = utils.load_features(
    engine=engine,
    df=df,
    aud_query=aud_query,
    date=feature_date,
    fav_omni_features_table=fav_omni_features_table_month,
    preperiod_months=preperiod_months,
)

print('After features:', df.shape)
display(df.head())


### 9. Prepare Dataset


In [ ]:
df = _prepare_dataset(df)
_validate_dataset(df, 'inference_dataset')

print('Prepared dataset shape:', df.shape)
print('Missing selected features:', sorted(set(features) - set(df.columns)))
display(df[features[:10]].head())


### 10. Data Quality Checks


In [ ]:
quality_report = pd.DataFrame({
    'metric': [
        'rows',
        'unique_contact_id',
        'feature_count',
        'missing_feature_count',
        'duplicated_contact_id',
    ],
    'value': [
        len(df),
        df['contact_id'].nunique(),
        len(features),
        len(set(features) - set(df.columns)),
        int(df['contact_id'].duplicated().sum()),
    ],
})

display(quality_report)

missing_report = (
    df[features]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename('missing_share')
    .reset_index(names='feature')
)

display(missing_report.head(30))


### 11. DAC Segments


In [ ]:
segment_report = (
    df.groupby('segment', dropna=False)
    .agg(rows=('contact_id', 'size'))
    .assign(share=lambda x: x['rows'] / x['rows'].sum())
    .sort_values('rows', ascending=False)
)

display(segment_report)


### 12. Outliers


In [ ]:
df['outlier'] = 0
for feature in features_for_outliers:
    if feature in df.columns and df[feature].notna().sum() > 0:
        df.loc[df[feature] > np.nanquantile(df[feature], 0.999), 'outlier'] = 1

outlier_share = df['outlier'].mean()
print('Outlier share:', outlier_share)
assert outlier_share <= 0.01, 'More than 1% of the audience was removed as outliers'

df = df[df['outlier'] == 0].drop(columns=['outlier']).reset_index(drop=True)
_validate_dataset(df, 'inference_dataset_without_outliers')

print('Final inference dataset shape:', df.shape)


### 13. Save Dataset


In [ ]:
utils.save_df_to_s3(df, s3, input_bucket, input_prefix)

objects = utils.list_s3_objects(s3, input_bucket, input_prefix)
print('Saved objects:', len(objects))
for obj in objects[:20]:
    print(obj)


### 14. Drop Temporary GP Tables


In [ ]:
for table in [aud_table_month, fav_omni_features_table_month]:
    try:
        utils.execute_query(engine, f'drop table if exists {table}')
        print('Dropped:', table)
    except Exception:
        logging.exception(f'Failed to drop temporary table {table}')

Path('df_cache.parquet').unlink(missing_ok=True)
